<a href="https://colab.research.google.com/github/detektor777/colab_list_image/blob/main/codiff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ##**Install** { display-mode: "form" }
import os, shutil, time

%cd /content

if not os.path.isdir('/content/CODiff'):
    !git clone --depth 1 https://github.com/jp-guo/CODiff.git

%cd /content/CODiff

# Install exact versions of dependencies
!pip install -q diffusers==0.25.0 transformers==4.37.2 peft==0.9.0 accelerate einops gdown opencv-python matplotlib tqdm huggingface_hub==0.25.2

import torch
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

# --- Patches ------------------------------------------------------------------
vaehook_path = 'diffusion/my_utils/vaehook.py'
if os.path.exists(vaehook_path):
    src = open(vaehook_path, encoding='utf-8').read()
    if 'from . import devices' not in src:
        src = src.replace('import devices', 'from . import devices')
        open(vaehook_path, 'w', encoding='utf-8').write(src)
        print('Patch applied: vaehook.py -> import devices fixed.')

# --- SD 2.1 Base Model Weights -----------------------------------------------
from huggingface_hub import snapshot_download

os.makedirs('model_zoo/sd-2-1-base', exist_ok=True)
if not os.path.exists('model_zoo/sd-2-1-base/model_index.json'):
    print('Downloading SD 2.1 weights (diffusers format) from Manojb/stable-diffusion-2-1-base mirror...')

    # Automatic resume upon connection drop (IncompleteRead)
    max_retries = 5
    for attempt in range(max_retries):
        try:
            snapshot_download(
                repo_id='Manojb/stable-diffusion-2-1-base',
                local_dir='model_zoo/sd-2-1-base',
                allow_patterns=['vae/*', 'unet/*', 'scheduler/*', 'model_index.json']
            )
            print('Base SD 2.1 model downloaded successfully.')
            break
        except Exception as e:
            print(f'\n⚠️ Connection drop (attempt {attempt + 1}/{max_retries}). HF servers might be overloaded.')
            if attempt < max_retries - 1:
                print('⏳ Resuming download in 3 seconds (progress cached)...')
                time.sleep(3)
            else:
                raise e

# --- CODiff and CaVE Weights --------------------------------------------------
import gdown

print('\n⚠️ Downloading CODiff and CaVE weights from Google Drive...')
CODIFF_GDRIVE_ID = "1SHZQs8fu3K419jlENoY8qdTKbjAOL0i7"
CAVE_GDRIVE_ID = "1SZo5UMSMEcYy4Wr-OIQgTk8hUXiFMIbK"

os.makedirs('model_zoo/codiff', exist_ok=True)

# Download only if files do not exist yet
if not os.path.exists('model_zoo/codiff/adapter_model.bin'):
    gdown.download(f'https://drive.google.com/uc?id={CODIFF_GDRIVE_ID}', 'model_zoo/codiff/adapter_model.bin', quiet=False)
else:
    print('✅ CODiff weights already downloaded.')

if not os.path.exists('model_zoo/cave.pth'):
    gdown.download(f'https://drive.google.com/uc?id={CAVE_GDRIVE_ID}', 'model_zoo/cave.pth', quiet=False)
else:
    print('✅ CaVE weights already downloaded.')

print('\n✅ Setup completed! Weights downloaded and dependencies installed successfully.')

In [ ]:
#@title ##**Upload Images** { display-mode: "form" }
import os
import shutil
from google.colab import files

upload_folder = './inputs/user_upload'
result_folder = './results/user_upload'

if os.path.isdir(upload_folder):
    shutil.rmtree(upload_folder)
if os.path.isdir(result_folder):
    shutil.rmtree(result_folder)

os.makedirs(upload_folder, exist_ok=True)
os.makedirs(result_folder, exist_ok=True)

# Upload images — you can select multiple files at once
uploaded = files.upload()
for filename in uploaded.keys():
    dst_path = os.path.join(upload_folder, filename)
    print(f'Moved {filename} -> {dst_path}')
    shutil.move(filename, dst_path)

print(f'\n{len(uploaded)} image(s) uploaded to {upload_folder}')

In [ ]:
#@title ##**Run** { display-mode: "form" }
import os, gc, torch, psutil, time

start_time_main = time.time()

# --- Prevent working directory loss after Restart session --------------
if os.path.basename(os.getcwd()) != 'CODiff':
    if os.path.isdir('/content/CODiff'):
        os.chdir('/content/CODiff')
        print(f'⚠️ Working directory was reset. Switched to {os.getcwd()}')
    else:
        raise RuntimeError('Folder /content/CODiff not found. Please run the "Install" cell.')

# --- Prevent upload_folder/result_folder variables loss ---------------
if 'upload_folder' not in globals():
    upload_folder = './inputs/user_upload'
    print(f'⚠️ upload_folder was not defined. Using: {upload_folder}')
if 'result_folder' not in globals():
    result_folder = './results/user_upload'
    print(f'⚠️ result_folder was not defined. Using: {result_folder}')

if not os.path.isdir(upload_folder) or not os.listdir(upload_folder):
    raise RuntimeError(f'No images found in {upload_folder}. Please run "Upload Images".')
os.makedirs(result_folder, exist_ok=True)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
torch.cuda.empty_cache()
print(f'Free RAM: {psutil.virtual_memory().available/1024**3:.2f} GiB')

# --- Generate custom inference script ---
inference_script = """
import os
import sys
import argparse
import gc
import json
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
import torchvision.transforms as transforms

# --- HUGGINGFACE_HUB COMPATIBILITY PATCH FOR DIFFUSERS 0.25.0 ---
import huggingface_hub
if not hasattr(huggingface_hub, 'cached_download'):
    huggingface_hub.cached_download = huggingface_hub.hf_hub_download
# -----------------------------------------------------------------

sys.path.append('diffusion')
sys.path.append('diffusion/my_utils')

from diffusion.codiff import CODiff_test
from cave.cave import CaVE
from diffusion.my_utils.wavelet_color_fix import adain_color_fix

# Replicate the default configuration Namespace settings from the official repository
class ModelArgs:
    def __init__(self, sd_path, codiff_path):
        self.pretrained_model = sd_path

        # Check if the path is a folder; automatically append the file name if so
        if os.path.isdir(codiff_path):
            self.codiff_path = os.path.join(codiff_path, "adapter_model.bin")
        else:
            self.codiff_path = codiff_path

        self.mixed_precision = "fp16"
        self.merge_and_unload_lora = False
        self.vae_decoder_tiled_size = 224
        self.vae_encoder_tiled_size = 1024
        self.latent_tiled_size = 96
        self.latent_tiled_overlap = 32

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--input', type=str, required=True)
    parser.add_argument('--output', type=str, required=True)
    parser.add_argument('--sd_path', type=str, default='model_zoo/sd-2-1-base')
    parser.add_argument('--codiff_path', type=str, default='model_zoo/codiff')
    parser.add_argument('--cave_path', type=str, default='model_zoo/cave.pth')
    args = parser.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("Loading official CODiff_test model...")
    model_args = ModelArgs(args.sd_path, args.codiff_path)
    model = CODiff_test(model_args)

    print("Loading official CaVE model...")
    cave = CaVE(in_nc=3, out_nc=3, nc=[64, 128, 256, 512], nb=4, act_mode='BR')
    if os.path.exists(args.cave_path):
        checkpoint = torch.load(args.cave_path, map_location='cpu')

        if isinstance(checkpoint, dict):
            if 'state_dict' in checkpoint:
                state_dict = checkpoint['state_dict']
            elif 'model' in checkpoint:
                state_dict = checkpoint['model']
            else:
                state_dict = checkpoint
        else:
            state_dict = checkpoint

        clean_state_dict = {}
        for k, v in state_dict.items():
            name = k[7:] if k.startswith('module.') else k
            clean_state_dict[name] = v

        cave.load_state_dict(clean_state_dict, strict=True)
        print("✅ CaVE weights loaded successfully.")
    cave.to(device).eval()

    images = sorted([f for f in os.listdir(args.input) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))])

    for img_name in tqdm(images, desc="Processing images"):
        img_path = os.path.join(args.input, img_name)
        img_H = Image.open(img_path).convert('RGB')

        # Dimensions must be multiples of 8 for VAE compatibility
        new_width = img_H.width - img_H.width % 8
        new_height = img_H.height - img_H.height % 8
        img_H = img_H.resize((new_width, new_height), Image.LANCZOS)

        # Prepare data matching official pipeline (lines 99-110 in main_test_codiff.py)
        img_L_np = np.array(img_H) # Original RGB NumPy array

        lq_raw = transforms.ToTensor()(img_H).unsqueeze(0).to(device) # lq_raw [0, 1] range
        lq = lq_raw * 2 - 1 # lq [-1, 1] range

        with torch.no_grad():
            # 1. Get visual embedding (CaVE expects [0, 1] raw tensor)
            visual_embedding = cave.get_visual_embedding(lq_raw)

            # 2. Run the official CODiff_test wrapper (does forward UNet and tiled VAE decode)
            img_E_tensor = model(lq, visual_embedding)

            # 3. Decode reconstructed tensor to PIL image (line 118 in official script)
            img_E = transforms.ToPILImage()(img_E_tensor[0].cpu() * 0.5 + 0.5)

        # 4. Align color using native AdaIN fix (line 120 in official script)
        try:
            # target = PIL image, source = original NumPy RGB array
            img_E_fixed = adain_color_fix(target=img_E, source=img_L_np)
            if not isinstance(img_E_fixed, Image.Image):
                img_E_fixed = Image.fromarray(img_E_fixed)
        except Exception as e:
            print(f"⚠️ AdaIN color fix skipped for {img_name}: {e}. Keeping original colors.")
            img_E_fixed = img_E

        # Resize back to original size if it was resized
        if img_E_fixed.size != (img_H.width, img_H.height):
            img_E_fixed = img_E_fixed.resize((img_H.width, img_H.height), Image.LANCZOS)

        img_E_fixed.save(os.path.join(args.output, img_name))

if __name__ == "__main__":
    main()
"""

with open('inference_colab.py', 'w', encoding='utf-8') as f:
    f.write(inference_script)

# Run inference
!python inference_colab.py --input {upload_folder} --output {result_folder}

# --- Calculate and display execution stats ----------------------------------
elapsed_time = time.time() - start_time_main
m, s = divmod(elapsed_time, 60)
h, m = divmod(m, 60)

time_str = ""
if h > 0:
    time_str += f"{int(h)}h "
if m > 0 or h > 0:
    time_str += f"{int(m)}m "
time_str += f"{s:.1f}s"

print(f'\n✅ Done! Processed images -> {result_folder}')
print(f'⏱️ Total execution time: {time_str}')
print(f'🖥️ RAM available after run: {psutil.virtual_memory().available/1024**3:.2f} GiB')

In [ ]:
#@title ##**Visualization** { display-mode: "form" }
import os
import base64
from io import BytesIO
import PIL.Image
from IPython.display import HTML, display

def is_image_file(filename):
    image_extensions = {'.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff'}
    return os.path.splitext(filename.lower())[1] in image_extensions

def resize_image_maintain_aspect(image, max_width, target_height=None):
    width, height = image.size
    if width > max_width:
        new_height = int(height * max_width / width)
        image = image.resize((max_width, new_height), PIL.Image.Resampling.LANCZOS)
    if target_height is not None and image.size[1] != target_height:
        new_width = int(image.size[0] * target_height / image.size[1])
        image = image.resize((new_width, target_height), PIL.Image.Resampling.LANCZOS)
    return image

def image_to_base64(image):
    buffered = BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

def find_images(root):
    found = []
    for dirpath, _, fnames in os.walk(root):
        for fname in sorted(fnames):
            if is_image_file(fname):
                found.append(os.path.join(dirpath, fname))
    return sorted(found)

filenames_input = sorted([f for f in os.listdir(upload_folder) if is_image_file(f)])
paths_output = find_images(result_folder)

if not filenames_input or not paths_output:
    print(f'Error: no images found in {upload_folder} or {result_folder}.')
else:
    for filename, path_output in zip(filenames_input, paths_output):
        try:
            image_before = PIL.Image.open(os.path.join(upload_folder, filename))
            image_after = PIL.Image.open(path_output)

            max_width = min(image_after.size[0], 1000)
            image_after = resize_image_maintain_aspect(image_after, max_width)
            target_height = image_after.size[1]
            image_before = resize_image_maintain_aspect(image_before, max_width, target_height)

            if image_before.mode != 'RGB':
                image_before = image_before.convert('RGB')
            if image_after.mode != 'RGB':
                image_after = image_after.convert('RGB')

            before_base64 = image_to_base64(image_before)
            after_base64 = image_to_base64(image_after)

            html_code = f"""
            <div style="position: relative; width: {image_after.size[0]}px; height: {image_after.size[1]}px; margin-bottom: 20px;">
                <div style="position: relative; width: 100%; height: 100%; overflow: hidden;">
                    <img src="data:image/png;base64,{before_base64}" style="position: absolute; width: 100%; height: 100%;">
                    <div style="position: absolute; width: 100%; height: 100%; overflow: hidden; clip-path: inset(0 0 0 50%);">
                        <img src="data:image/png;base64,{after_base64}" style="position: absolute; width: 100%; height: 100%;">
                    </div>
                </div>
                <div class="slider" style="position: absolute; top: 0; bottom: 0; width: 4px; background: white; cursor: ew-resize; left: 50%; box-shadow: 0 0 5px rgba(0,0,0,0.5);">
                    <div style="position: absolute; top: 50%; transform: translateY(-50%); width: 20px; height: 20px; background: white; border-radius: 50%; left: -8px;"></div>
                </div>
                <div style="position: absolute; top: 8px; left: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">Before</div>
                <div style="position: absolute; top: 8px; right: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">After</div>
            </div>
            <script>
                document.querySelectorAll('.slider').forEach(slider => {{
                    let isDragging = false;
                    const container = slider.parentElement.querySelector('div:nth-child(1)');
                    const clipDiv = container.querySelector('div');

                    slider.addEventListener('mousedown', (e) => {{ isDragging = true; e.preventDefault(); }});
                    document.addEventListener('mouseup', () => {{ isDragging = false; }});
                    document.addEventListener('mousemove', (e) => {{
                        if (!isDragging) return;
                        const rect = container.getBoundingClientRect();
                        let x = e.clientX - rect.left;
                        if (x < 0) x = 0;
                        if (x > rect.width) x = rect.width;
                        const percentage = (x / rect.width) * 100;
                        slider.style.left = percentage + '%';
                        clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                    }});

                    slider.addEventListener('touchstart', (e) => {{ isDragging = true; e.preventDefault(); }});
                    document.addEventListener('touchend', () => {{ isDragging = false; }});
                    document.addEventListener('touchmove', (e) => {{
                        if (!isDragging) return;
                        const rect = container.getBoundingClientRect();
                        let x = e.touches[0].clientX - rect.left;
                        if (x < 0) x = 0;
                        if (x > rect.width) x = rect.width;
                        const percentage = (x / rect.width) * 100;
                        slider.style.left = percentage + '%';
                        clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                    }});
                }});
            </script>
            """

            display(HTML(html_code))

        except Exception as e:
            print(f'Error processing {filename} and {path_output}: {e}')

In [ ]:
#@title ##**Download Results** { display-mode: "form" }
import os
from google.colab import files

# Fallback to the default folder if the variable was lost due to session restart
if 'result_folder' not in globals():
    result_folder = './results/user_upload'

if not os.path.exists(result_folder):
    print(f"⚠️ Error: Directory '{result_folder}' does not exist.")
else:
    # Filter only files (ignoring subdirectories)
    processed_files = [f for f in os.listdir(result_folder) if os.path.isfile(os.path.join(result_folder, f))]

    if len(processed_files) == 0:
        print("⚠️ No processed images found to download.")
    elif len(processed_files) == 1:
        # Download the single file directly
        single_file_path = os.path.join(result_folder, processed_files[0])
        print(f"📥 Downloading single image directly: {processed_files[0]}")
        files.download(single_file_path)
    else:
        # Package multiple files into a ZIP archive
        zip_filename = 'CODiff_result.zip'
        if os.path.exists(zip_filename):
            os.remove(zip_filename)

        print(f"📥 Packaging {len(processed_files)} images into a ZIP archive...")
        os.system(f"zip -r -j {zip_filename} {result_folder}/*")
        files.download(zip_filename)